In [ ]:
import pandas as pd
url="https://raw.githubusercontent.com/AdyashaNayak16/Prompt_Injection_Detector_final/main/NOTEBOOKS/full_dataset.csv"
df=pd.read_csv(url)
print(df.shape)
print(df["label"].value_counts())

['.config', 'sample_data']


In [ ]:
from sklearn.model_selection import train_test_split
X=df["text"].tolist()
y=df["label"].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts, max_len=128):
    return tokenizer(
        texts,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

train_enc = tokenize(X_train)
val_enc = tokenize(X_val)
test_enc = tokenize(X_test)

print("Tokenization done!")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class InjectionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = InjectionDataset(train_enc, y_train)
val_dataset = InjectionDataset(val_enc, y_val)
test_dataset = InjectionDataset(test_enc, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

print("Datasets ready!")

In [ ]:
from transformers import DistilBertForSequenceClassification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)
model.to(device)
print("Model loaded!")

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} — Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import torch

model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        probs = torch.softmax(logits, dim=1)[:, 1]
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

print(classification_report(all_labels, all_preds))
print("ROC-AUC:", roc_auc_score(all_labels, all_probs))

In [ ]:
from huggingface_hub import login
import os

login(token=os.environ.get("hf_token"))

model.push_to_hub("prompt-injection-detector")
tokenizer.push_to_hub("prompt-injection-detector")
print("Pushed to HuggingFace!")